# TCM-FuzzyWiki V5.0 · Colab（MiniMax-M3 / Azure Kimi-K2.6，多并发 + 断点续跑）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pariskang/TCM-FuzzyWiki/blob/main/colab/TCM_FuzzyWiki_MiniMax_M3_Colab.ipynb)

完整复现 `https://github.com/pariskang/TCM-FuzzyWiki`，用 **MiniMax-M3** 或 **Azure Kimi-K2.6** 做 observation-first 抽取，再由仓库内**确定性 pipeline** 复算 membership / 共现 / 规则 / Larsen 推理 / Mamdani / 三层聚合 / 关系网络 / Markdown Wiki / validation / audit / manifest。

本 notebook 覆盖仓库**全部功能**：`run-demo`、`normalize-input`、`doctor`、`build-llm`（OpenAI / Anthropic 两种 MiniMax-M3 协议，以及 **Azure OpenAI 兼容**协议，如 Kimi-K2.5 部署）、`assess`、`roleplay-score`、`calibrate`、关系网络导入、金标准评估与专家规则升格。

- **三种接入**：`openai`（MiniMax `/v1`）、`anthropic`（MiniMax `/anthropic`）、`azure`（Azure `/openai/v1`，Bearer 鉴权，部署名即模型名）。只需改第 3 格 `PROVIDER`。
- **多并发**抽取（`WORKERS`）+ **chunk 级断点续跑**：Colab 断线 / 限流 / 崩溃后，重跑同一 cell 只补抽失败或缺失 chunk。
- **实时进度条 + 实时保存**：`build-llm` 抽取时显示 tqdm 进度条（已完成/总块数、成功/失败、抽到的条数），并每 ~20s 把已抽 observation 落盘到 `extraction/observations_checkpoint.csv`，进度写入 `extraction/progress.json`，随时可查、断了不丢。
- **中英文列名兼容** + 输入预检：原始 XLSX 用 `书名/篇名/原文/朝代` 等中文列也能直接跑；若所有正文为空，会 fail-fast 给修复指引。
- **不限制抽取数量**：`MAX_OBS_PER_CHUNK=0` 让模型穷尽每块的实体，共现关系挖掘 `top_k_patterns=0` 保留全部统计显著模式（不再截断为前 N 条）。
- observation ID 按输入顺序确定性分配，**结果可复现**。

> 安全：**不要**把 API Key 写进 notebook。请用 Colab 左侧 🔑 Secrets 设置 `MINIMAX_API_KEY` 或 `AZURE_OPENAI_API_KEY`，或运行时手动输入。


## 1. 挂载 Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 克隆仓库并安装


In [ ]:
%cd /content
!rm -rf TCM-FuzzyWiki
!git clone https://github.com/pariskang/TCM-FuzzyWiki.git
%cd /content/TCM-FuzzyWiki
# 该分支已含 build-llm / normalize-input / azure provider；合并入 main 后此步可忽略
!git checkout claude/wonderful-keller-8nxrr6 || echo "using default branch"
!git rev-parse --short HEAD

!python -m pip install -q -e '.[dev]'
# 可选 SDK、JSON 修复、图分析（Anthropic 协议装了 anthropic 会自动走 SDK）
!python -m pip install -q --upgrade openai anthropic json-repair networkx tqdm

# 离线 demo + 单测，确认环境与确定性 pipeline 正常
!tcm-fuzzywiki run-demo --output /content/drive/MyDrive/TCM_FuzzyWiki_MiniMax_M3/_demo
!python -m pytest -q

## 3. 配置 API Key 与运行参数


In [ ]:
import os, getpass, datetime
from pathlib import Path

# ===== 选择 LLM 提供方 =====
# 'openai'    -> MiniMax-M3，OpenAI 协议   (https://api.minimaxi.com/v1)
# 'anthropic' -> MiniMax-M3，Anthropic 协议 (https://api.minimaxi.com/anthropic)
# 'azure'     -> Azure OpenAI 兼容 /openai/v1（如 Kimi-K2.5 部署，部署名即模型名）
PROVIDER = 'openai'

# Azure 专用参数（仅 PROVIDER='azure' 生效）：资源 endpoint 与部署名
AZURE_ENDPOINT = 'https://foste-mqufb5uv-swedencentral.cognitiveservices.azure.com/openai/v1'
AZURE_DEPLOYMENT = 'Kimi-K2.6'


def _load_secret(name, prompt):
    val = None
    try:
        from google.colab import userdata
        val = userdata.get(name)
    except Exception:
        val = None
    val = val or os.environ.get(name)
    if not val:
        val = getpass.getpass(prompt)
    return val


if PROVIDER == 'azure':
    # 手动输入 Azure API Key：在下方输入框粘贴 key 后按回车确认（不回显，不写入 notebook）
    os.environ['AZURE_OPENAI_API_KEY'] = getpass.getpass('请输入 Azure API Key 后回车确认（不回显）：')
    os.environ['AZURE_OPENAI_ENDPOINT'] = AZURE_ENDPOINT
    os.environ['AZURE_OPENAI_DEPLOYMENT'] = AZURE_DEPLOYMENT
    MODEL = AZURE_DEPLOYMENT
    BASE_URL = AZURE_ENDPOINT
    print('Azure Key loaded:', bool(os.environ.get('AZURE_OPENAI_API_KEY')),
          '| endpoint:', AZURE_ENDPOINT, '| deployment:', AZURE_DEPLOYMENT)
else:
    os.environ['MINIMAX_API_KEY'] = _load_secret('MINIMAX_API_KEY', '请输入 MiniMax API Key（不显示）：')
    MODEL = 'MiniMax-M3'
    BASE_URL = ''  # 留空走 provider 默认 base URL
    print('MiniMax Key loaded:', bool(os.environ.get('MINIMAX_API_KEY')))

# ===== 可调参数 =====
WORKERS = 6                # 并发；遇到 429/限流就调小（如 2-3）
CHUNK_CHARS = 2400         # 长章节切块大小
MAX_OBS_PER_CHUNK = 0      # 0 = 不限制：每块穷尽抽取所有实体/关系（默认）。设正数则按上限截断
THINKING = 'disabled'      # 仅 MiniMax openai/anthropic 生效；azure 会自动忽略该字段

# 输入古籍表格（任意列名，下一步会自动规范化）
RAW_INPUT = Path('/content/drive/MyDrive/中医古籍指令数据/book_chapters.xlsx')

# 绝对路径，避免 cell 间 cwd 变化导致找不到 config
CONFIG = '/content/TCM-FuzzyWiki/configs/tcm_fuzzywiki.yaml'

RUN_TAG = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
DRIVE_ROOT = Path('/content/drive/MyDrive/TCM_FuzzyWiki_MiniMax_M3')
OUTPUT_DIR = DRIVE_ROOT / f'run_{RUN_TAG}'
SMOKE_DIR = OUTPUT_DIR / '_smoke'
NORMALIZED = OUTPUT_DIR / 'input' / 'chapters.normalized.xlsx'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert RAW_INPUT.exists(), f'找不到输入文件：{RAW_INPUT}'
print('PROVIDER:', PROVIDER, '| MODEL:', MODEL, '| OUTPUT_DIR:', OUTPUT_DIR)

## 4. 规范化任意表格为推荐字段

把任意中英文列名映射为推荐 schema、自动猜测正文列、补默认 metadata、输出可审计的列映射报告。

**重要**：从下一步起所有命令都使用规范化后的 `NORMALIZED_CSV`，**不要再传原始 XLSX**——否则若列名识别失败会得到 0 个 observation。


In [ ]:
!tcm-fuzzywiki normalize-input \
  --input "$RAW_INPUT" \
  --output "$NORMALIZED" \
  --report "$OUTPUT_DIR/input/column_mapping_report.json"

NORMALIZED_CSV = NORMALIZED.with_suffix('.csv')
import pandas as pd
print('规范化 CSV：', NORMALIZED_CSV)
display(pd.read_csv(NORMALIZED_CSV).head(3))

## 5. 输入健康检查（避免 88883 个 no_chunks 那种翻车）

核对正文非空率与 token 量级；若几乎全空，立刻停下来回去看 cell 4 的列映射。


In [ ]:
import pandas as pd
df = pd.read_csv(NORMALIZED_CSV)
n = len(df)
n_with_text = int((df['text_original'].fillna('').astype(str).str.strip().str.len() > 0).sum())
avg_chars = float(df['text_original'].fillna('').astype(str).str.len().mean())
print(f'规范化后行数: {n}')
print(f'正文非空行数: {n_with_text} ({n_with_text / max(n, 1):.1%})')
print(f'正文平均字符数: {avg_chars:.1f}')
assert n_with_text > 0, '所有行的 text_original 都为空：请回到 cell 4 检查列映射，或在 column_mapping_report.json 中确认 text_original 来源列正确。'
print('粗略 token 估算（按 1.5 char/token）：', int(df['text_original'].fillna('').astype(str).str.len().sum() / 1.5))

## 6. 配置/数据体检（doctor）


In [ ]:
!tcm-fuzzywiki doctor --config "$CONFIG" --input "$NORMALIZED_CSV" 

## 7. 小样本试跑（先验证 3 章，写入独立 _smoke 目录）


In [ ]:
!tcm-fuzzywiki build-llm \
  --input "$NORMALIZED_CSV" \
  --config "$CONFIG" \
  --output "$SMOKE_DIR" \
  --provider "$PROVIDER" \
  --model "$MODEL" \
  --base-url "$BASE_URL" \
  --workers 3 \
  --chunk-chars $CHUNK_CHARS \
  --max-observations-per-chunk $MAX_OBS_PER_CHUNK \
  --thinking "$THINKING" \
  --limit 3

!cat "$SMOKE_DIR/summary.txt"

# 验收试跑：obs > 0 才进入全量
import pandas as pd, json
from pathlib import Path
smoke = pd.read_csv(Path(SMOKE_DIR) / 'data' / 'observations.csv') if (Path(SMOKE_DIR)/'data'/'observations.csv').stat().st_size > 4 else pd.DataFrame()
print('试跑 observation 数:', len(smoke))
assert len(smoke) > 0, '试跑没有产出 observation：检查 _smoke/extraction/llm_errors.csv 与原始日志，确认 API Key / base URL / model（Azure 为部署名）都正确。'

## 8. 全量运行（多并发 + 断点续跑）

直接**重跑本 cell 即可续跑**：`success` 的 chunk 自动跳过，只补抽失败/缺失 chunk。
续跑时切勿修改 `CHUNK_CHARS` 或更换输入文件（manifest 会拒绝错配）。


> 默认 `MAX_OBS_PER_CHUNK=0`（不限制）：模型会尽量抽全每块的实体，token 消耗与耗时随之上升；若遇到限流或预算紧张，可在第 3 格把它改成正整数（如 12）按上限截断。

In [ ]:
!tcm-fuzzywiki build-llm \
  --input "$NORMALIZED_CSV" \
  --config "$CONFIG" \
  --output "$OUTPUT_DIR" \
  --provider "$PROVIDER" \
  --model "$MODEL" \
  --base-url "$BASE_URL" \
  --workers $WORKERS \
  --chunk-chars $CHUNK_CHARS \
  --max-observations-per-chunk $MAX_OBS_PER_CHUNK \
  --thinking "$THINKING"

!cat "$OUTPUT_DIR/summary.txt" 

## 9. 实时进度 / 断点状态

抽取时第 8 格会**直接显示 tqdm 进度条**；本格用于另开一个 cell 随时查看 `progress.json` 快照与断点明细（抽取中/中断后都可运行）。

In [ ]:
import json
import pandas as pd
from pathlib import Path

ext = Path(OUTPUT_DIR) / 'extraction'

# 进度快照（build-llm 实时写入 progress.json）
pj = ext / 'progress.json'
if pj.exists():
    p = json.loads(pj.read_text())
    pct = p.get('percent_complete', 0.0)
    bar = '█' * int(pct // 5) + '░' * (20 - int(pct // 5))
    print(f"[{bar}] {pct:5.1f}%  阶段={p.get('phase')}")
    print(f"总块数={p.get('total_chunks')} | 续跑跳过={p.get('skipped_resumed')} | "
          f"本次完成={p.get('completed_this_run')}/{p.get('pending_this_run')} | "
          f"失败={p.get('failed_this_run')} | 已抽 observation≈{p.get('observation_count')}")
    print('快照时间:', p.get('timestamp'))
    try:
        from tqdm.auto import tqdm  # 可视化进度条快照
        done = int(p.get('skipped_resumed', 0)) + int(p.get('completed_this_run', 0))
        t = tqdm(total=int(p.get('total_chunks', 0)) or 1, desc='extract(snapshot)')
        t.update(done); t.refresh(); t.close()
    except Exception:
        pass
else:
    print('progress.json 尚未生成（抽取还没开始）')

# 文本状态 + 断点明细
ls = ext / 'live_status.txt'
print('\n--- live_status.txt ---')
print(ls.read_text() if ls.exists() else '尚未开始')
sp = ext / 'source_progress.csv'
if sp.exists() and sp.stat().st_size > 4:
    df = pd.read_csv(sp)
    display(df['status'].value_counts(dropna=False))
    display(df[df['status'].isin(['error', 'partial_success'])].head(20))

## 10. 关键产物检查（带空文件安全护栏）


In [ ]:
import pandas as pd, json
from pathlib import Path

def safe_read_head(p: Path, n: int = 10):
    if not p.exists() or p.stat().st_size <= 4:
        return f'(空或不存在: {p.name})'
    try:
        return pd.read_csv(p).head(n)
    except pd.errors.EmptyDataError:
        return f'(无列: {p.name})'

data = Path(OUTPUT_DIR) / 'data'
for name in ['observations.csv','memberships.csv','candidate_patterns.csv','rules.csv',
             'inference_results.csv','aggregation_results.csv','mamdani_results.csv',
             'relation_edges.csv','evaluation_results.csv','validation_report.csv',
             'completion_assessment.csv','run_manifest.json']:
    p = data / name
    size = p.stat().st_size if p.exists() else 0
    print(f"{'OK  ' if size > 4 else 'EMPTY'} {size:>10}  {name}")

display(safe_read_head(data / 'observations.csv'))
man = json.loads((data / 'run_manifest.json').read_text())
print('provider:', man.get('llm_provider'), '| model:', man.get('llm_model'))
print('extraction_report:', json.dumps(man.get('extraction_report', {}), ensure_ascii=False))
print('Wiki 首页：', Path(OUTPUT_DIR) / 'wiki' / 'index.md')

## 11. 完整性评估（assess）

给出该次构建是 `formal_ready` / `research_ready_with_caveats` / `not_ready` 的明确判定。


In [ ]:
!tcm-fuzzywiki assess --output "$OUTPUT_DIR" 

## 12. 关系网络导入 NetworkX（可选；可进一步导出 Gephi / Neo4j）


In [ ]:
import pandas as pd, networkx as nx
from collections import Counter
from pathlib import Path
data = Path(OUTPUT_DIR) / 'data'
nodes = pd.read_csv(data / 'relation_nodes.csv') if (data/'relation_nodes.csv').stat().st_size > 4 else pd.DataFrame()
edges = pd.read_csv(data / 'relation_edges.csv') if (data/'relation_edges.csv').stat().st_size > 4 else pd.DataFrame()
G = nx.DiGraph()
for _, r in nodes.iterrows():
    G.add_node(r['node_id'], node_type=r.get('node_type'), label=r.get('label'))
for _, r in edges.iterrows():
    G.add_edge(r['source'], r['target'], edge_type=r.get('edge_type'))
print(f'节点 {G.number_of_nodes()}，边 {G.number_of_edges()}')
print('节点类型分布：', dict(Counter(nx.get_node_attributes(G, 'node_type').values())))
# 如需 Gephi：nx.write_graphml(G, str(Path(OUTPUT_DIR)/'relation_network.graphml'))

## 13. LLM 分角色专家校准（离线 demo，无需 API）

让"现代循证医学 / 中医 / 古文字"三类角色对语言变量映射打分，生成 calibrated YAML（`calibration_source: llm_roleplay_panel`，便于后续人工复核）。
真实 LLM 打分需 Azure 适配器：把 `--offline-demo` 换成 `--azure-llm` 并配置 Azure 环境变量。


In [ ]:
!tcm-fuzzywiki roleplay-score \
  --config "$CONFIG" \
  --output-scores "$OUTPUT_DIR/calibration/roleplay_scores.csv" \
  --output-config "$OUTPUT_DIR/calibration/linguistic_values.roleplay_calibrated.yaml" \
  --report "$OUTPUT_DIR/calibration/roleplay_report.csv" \
  --offline-demo

import pandas as pd
display(pd.read_csv(f"{OUTPUT_DIR}/calibration/roleplay_report.csv").head(10))

## 14. 专家 membership 校准（calibrate）

用多专家评分把 bootstrap 先验校准为 `expert_reviewed`，并输出含 ICC、p5/p95 的完整 config。
下面用一个**示例**评分表演示；真实使用请替换为人工/多专家 CSV（列：`term,variable,fuzzy_set,expert_id,score`）。


In [ ]:
import pandas as pd
from pathlib import Path
cal_dir = Path(OUTPUT_DIR) / 'calibration'
cal_dir.mkdir(parents=True, exist_ok=True)
EXPERT_CSV = cal_dir / 'expert_membership_scores.csv'
CALIBRATED_CONFIG = cal_dir / 'linguistic_values.calibrated.yaml'
pd.DataFrame([
    {'term': '冷痛', 'variable': 'cold_property', 'fuzzy_set': 'high', 'expert_id': 'EXP_001', 'score': 0.90},
    {'term': '冷痛', 'variable': 'cold_property', 'fuzzy_set': 'high', 'expert_id': 'EXP_002', 'score': 0.86},
    {'term': '冷痛', 'variable': 'cold_property', 'fuzzy_set': 'high', 'expert_id': 'EXP_003', 'score': 0.88},
]).to_csv(EXPERT_CSV, index=False, encoding='utf-8-sig')

!tcm-fuzzywiki calibrate --config "$CONFIG" --expert-scores "$EXPERT_CSV" \
  --output-config "$CALIBRATED_CONFIG" --report "$cal_dir/calibration_report.csv"

display(pd.read_csv(f"{cal_dir}/calibration_report.csv"))
print('校准后可用该 config 重跑：build-llm ... --config', CALIBRATED_CONFIG)

## 15. 可选：金标准评估 + 专家规则升格 + 切换提供方（Anthropic / Azure Kimi-K2.5）

**金标准评估**：把 FCR/CRP/MIC/SMB/FIA 所需的专家 CSV 放进一个目录，重跑加 `--gold-dir`（缺金标准时系统输出 `needs_gold_standard` 模板，不会伪造结论）。

**专家规则升格**：审核 `data/expert_rule_review.csv` 后整理为 `rules.csv`，重跑加 `--rules-csv` 把候选共现模式升格为正式规则。

**切换 Anthropic 协议**：把第 3 格 `PROVIDER` 改为 `'anthropic'` 重跑第 7、8 格即可，base URL 自动指向 `/anthropic`。

**切换 Azure Kimi-K2.6**：把第 3 格 `PROVIDER` 改为 `'azure'`，并按你的资源填好 `AZURE_ENDPOINT`（默认 `https://foste-mqufb5uv-swedencentral.cognitiveservices.azure.com/openai/v1`）、`AZURE_DEPLOYMENT`（部署名即模型名，默认 `Kimi-K2.6`），运行第 3 格时**手动输入 API Key 并回车确认**，再重跑第 7、8 格即可。Azure 用 OpenAI 兼容的 `/openai/v1` 接口与 Bearer 鉴权，`THINKING` 字段会自动忽略。

下面给出可选模板（按需取消注释并填好路径）：

In [ ]:
# GOLD_DIR = '/content/drive/MyDrive/.../gold_csv_dir'   # 含 expert_memberships.csv 等
# RULES_CSV = '/content/drive/MyDrive/.../rules.csv'        # 专家审核后的规则
# !tcm-fuzzywiki build-llm \
#   --input "$NORMALIZED_CSV" --config "$CONFIG" --output "$OUTPUT_DIR" \
#   --provider "$PROVIDER" --model "$MODEL" --base-url "$BASE_URL" --workers $WORKERS \
#   --chunk-chars $CHUNK_CHARS --max-observations-per-chunk $MAX_OBS_PER_CHUNK \
#   --thinking "$THINKING" --gold-dir "$GOLD_DIR" --rules-csv "$RULES_CSV"

print('可选步骤模板；如需评估/规则升格请取消注释并填好路径。')
print('⚠️ 切勿把 API Key 写入 notebook 或提交到仓库。')